# CT Image Denoising with GAN

This notebook trains a GAN-based CT image denoising model using the DeepLesion dataset.
It includes a DnCNN baseline for comparison, and evaluates both using PSNR and SSIM.

**Dataset:** NIH DeepLesion Subset — add it via Kaggle Datasets before running.

**Steps:**
1. Cell 1 — GPU check
2. Cell 2 — Setup directories
3. Cell 3 — Write project files
4. Cell 4 — Train CNN baseline
5. Cell 5 — Train GAN
6. Cell 6 — Evaluate and plot results

**Key fix:** `cfg.py` is used instead of `config.py` to avoid collision with PyTorch's internal `config` module.

## Cell 1 — Check GPU

In [ ]:
import os
os.environ["TORCH_DISABLE_DYNAMO"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name}  |  VRAM: {vram:.1f} GB')
else:
    print('No GPU detected. Go to Settings -> Accelerator -> GPU (T4)')

## Cell 2 — Setup Directories

In [ ]:
import os, sys

PROJECT = '/kaggle/working/ct_denoising'
for d in ['data/images', 'checkpoints', 'results', 'models', 'training', 'testing', 'utils']:
    os.makedirs(f'{PROJECT}/{d}', exist_ok=True)

sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

DATA_DIR = '/kaggle/input/datasets/kmader/nih-deeplesion-subset/minideeplesion'
if not os.path.exists(DATA_DIR):
    DATA_DIR = f'{PROJECT}/data/images'
    print(f'Dataset not found — using {DATA_DIR}')
else:
    print(f'Dataset found: {DATA_DIR}')
print(f'Working dir: {os.getcwd()}')

## Cell 3 — Write Project Files

**Important:** We use `cfg.py` (not `config.py`) to avoid a name collision with PyTorch's internal `torch._dynamo.config` module, which caused the `AssertionError` in the original notebook.

In [ ]:
import os, textwrap

PROJECT = '/kaggle/working/ct_denoising'

def write(rel_path, src):
    full = os.path.join(PROJECT, rel_path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, 'w') as f:
        f.write(textwrap.dedent(src).strip() + '\n')

# ── cfg.py (renamed from config.py to avoid PyTorch internal collision) ────────
write('cfg.py', '''
    import os

    BASE_DIR       = '/kaggle/working/ct_denoising'
    DATA_DIR       = '/kaggle/input/datasets/kmader/nih-deeplesion-subset/minideeplesion'
    if not os.path.exists(DATA_DIR):
        DATA_DIR   = os.path.join(BASE_DIR, 'data', 'images')

    CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
    RESULTS_DIR    = os.path.join(BASE_DIR, 'results')
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR,    exist_ok=True)

    IMAGE_SIZE  = 256
    TEST_SPLIT  = 0.15
    VAL_SPLIT   = 0.10
    NUM_WORKERS = 2
    PIN_MEMORY  = True

    NOISE_TYPE      = "gaussian"
    GAUSSIAN_SIGMA  = 30.0
    POISSON_SCALE   = 0.1

    BATCH_SIZE   = 4
    NUM_CHANNELS = 1
    SEED         = 42

    # CNN baseline
    CNN_EPOCHS     = 60
    CNN_LR         = 1e-3
    CNN_LR_STEP    = 20
    CNN_LR_GAMMA   = 0.5
    CNN_SAVE_EVERY = 10
    CNN_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'cnn_best.pth')

    # GAN
    GAN_EPOCHS     = 55
    GAN_LR_G       = 1e-4
    GAN_LR_D       = 1e-4
    GAN_BETA1      = 0.9
    GAN_BETA2      = 0.999
    GAN_SAVE_EVERY = 10

    LAMBDA_ADV   = 1e-3
    LAMBDA_VGG   = 6e-3
    LAMBDA_PIXEL = 1.0

    GEN_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'generator_best.pth')
    DIS_CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'discriminator_best.pth')

    NUM_VISUAL_SAMPLES = 8
    RESULTS_CSV        = os.path.join(RESULTS_DIR, 'metrics.csv')
    COMPARISON_PLOT    = os.path.join(RESULTS_DIR, 'comparison.png')
''')

# ── models/__init__.py ────────────────────────────────────────────────────────
write('models/__init__.py', '''
    from .generator     import Generator
    from .discriminator import Discriminator
    from .cnn           import DenoisingCNN
''')

# ── models/generator.py ───────────────────────────────────────────────────────
write('models/generator.py', '''
    import torch.nn as nn

    class ResidualBlock(nn.Module):
        def __init__(self, channels=64):
            super().__init__()
            self.block = nn.Sequential(
                nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
                nn.BatchNorm2d(channels),
                nn.PReLU(),
                nn.Conv2d(channels, channels, 3, 1, 1, bias=False),
                nn.BatchNorm2d(channels),
            )
        def forward(self, x):
            return x + self.block(x)

    class Generator(nn.Module):
        def __init__(self, in_channels=1, out_channels=1, n_res_blocks=16, n_features=64):
            super().__init__()
            self.head = nn.Sequential(
                nn.Conv2d(in_channels, n_features, 9, 1, 4),
                nn.PReLU()
            )
            self.res_blocks = nn.Sequential(
                *[ResidualBlock(n_features) for _ in range(n_res_blocks)]
            )
            self.post_res = nn.Sequential(
                nn.Conv2d(n_features, n_features, 3, 1, 1, bias=False),
                nn.BatchNorm2d(n_features),
            )
            self.tail = nn.Sequential(
                nn.Conv2d(n_features, out_channels, 9, 1, 4),
                nn.Sigmoid()
            )

        def forward(self, x):
            head = self.head(x)
            body = self.post_res(self.res_blocks(head))
            return self.tail(head + body)
''')

# ── models/discriminator.py ───────────────────────────────────────────────────
write('models/discriminator.py', '''
    import torch.nn as nn

    def _disc_block(in_ch, out_ch, stride=1, bn=True):
        layers = [nn.Conv2d(in_ch, out_ch, 4, stride, 1, bias=not bn)]
        if bn:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        return nn.Sequential(*layers)

    class Discriminator(nn.Module):
        def __init__(self, in_channels=1, n_features=64):
            super().__init__()
            self.net = nn.Sequential(
                _disc_block(in_channels,        n_features,     stride=2, bn=False),
                _disc_block(n_features,         n_features * 2, stride=2),
                _disc_block(n_features * 2,     n_features * 4, stride=2),
                _disc_block(n_features * 4,     n_features * 8, stride=1),
                nn.Conv2d(n_features * 8, 1, 4, 1, 1),
            )
        def forward(self, x):
            return self.net(x)
''')

# ── models/cnn.py ─────────────────────────────────────────────────────────────
write('models/cnn.py', '''
    import torch
    import torch.nn as nn

    class DenoisingCNN(nn.Module):
        def __init__(self, in_channels=1, depth=17, n_features=64):
            super().__init__()
            layers = [
                nn.Conv2d(in_channels, n_features, 3, 1, 1, bias=True),
                nn.ReLU(inplace=True)
            ]
            for _ in range(depth - 2):
                layers += [
                    nn.Conv2d(n_features, n_features, 3, 1, 1, bias=False),
                    nn.BatchNorm2d(n_features),
                    nn.ReLU(inplace=True)
                ]
            layers.append(nn.Conv2d(n_features, in_channels, 3, 1, 1, bias=True))
            self.dncnn = nn.Sequential(*layers)
            self._init_weights()

        def _init_weights(self):
            for m in self.modules():
                if isinstance(m, nn.Conv2d):
                    nn.init.kaiming_normal_(m.weight)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.BatchNorm2d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

        def forward(self, x):
            residual = self.dncnn(x)
            return torch.clamp(x - residual, 0.0, 1.0)
''')

# ── training/__init__.py ──────────────────────────────────────────────────────
write('training/__init__.py', '')

# ── training/data_loader.py ───────────────────────────────────────────────────
write('training/data_loader.py', '''
import os, random, sys
from pathlib import Path
import numpy as np
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, '/kaggle/working/ct_denoising')
import cfg as config

class Resize:
    def __init__(self, size): self.size = size
    def __call__(self, img):  return img.resize(self.size, Image.BILINEAR)

class RandomHorizontalFlip:
    def __call__(self, img):
        return ImageOps.mirror(img) if random.random() > 0.5 else img

class RandomVerticalFlip:
    def __call__(self, img):
        return ImageOps.flip(img) if random.random() > 0.5 else img

class RandomRotation:
    def __init__(self, deg): self.deg = deg
    def __call__(self, img):
        return img.rotate(random.uniform(-self.deg, self.deg))

class ToTensor:
    def __call__(self, img):
        arr = np.array(img, dtype=np.float32) / 255.0
        if arr.ndim == 2:
            arr = arr[:, :, None]
        return torch.tensor(arr.transpose(2, 0, 1), dtype=torch.float32)

class Compose:
    def __init__(self, transforms): self.transforms = transforms
    def __call__(self, img):
        for t in self.transforms: img = t(img)
        return img

def add_gaussian_noise(tensor, sigma):
    noise = torch.randn_like(tensor) * (sigma / 255.0)
    return torch.clamp(tensor + noise, 0.0, 1.0)

def add_poisson_noise(tensor, scale=0.1):
    return torch.clamp(torch.poisson(tensor / scale) * scale, 0.0, 1.0)

def add_noise(tensor):
    if config.NOISE_TYPE == "gaussian":
        return add_gaussian_noise(tensor, config.GAUSSIAN_SIGMA)
    elif config.NOISE_TYPE == "poisson":
        return add_poisson_noise(tensor, config.POISSON_SCALE)
    elif config.NOISE_TYPE == "mixed":
        t = add_gaussian_noise(tensor, config.GAUSSIAN_SIGMA * 0.6)
        return add_poisson_noise(t, config.POISSON_SCALE)
    raise ValueError(f"Unknown NOISE_TYPE: {config.NOISE_TYPE}")

class CTDataset(Dataset):
    EXTENSIONS = {".png", ".jpg", ".jpeg", ".tiff", ".bmp"}

    def __init__(self, image_paths, augment=False):
        self.image_paths = image_paths
        aug_steps = ([RandomHorizontalFlip(), RandomVerticalFlip(), RandomRotation(10)]
                     if augment else [])
        self.transform = Compose([
            Resize((config.IMAGE_SIZE, config.IMAGE_SIZE)),
            *aug_steps,
            ToTensor(),
        ])

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("L")
        arr = np.clip(np.array(img, dtype=np.float32), 0, 4096) / 4096.0
        img = Image.fromarray((arr * 255).astype(np.uint8))
        clean = self.transform(img)
        noisy = add_noise(clean.clone())
        return noisy, clean

def get_dataloaders(data_dir=None, batch_size=None):
    data_dir   = data_dir   or config.DATA_DIR
    batch_size = batch_size or config.BATCH_SIZE

    all_paths = sorted([
        str(p) for p in Path(data_dir).rglob("*")
        if p.suffix.lower() in CTDataset.EXTENSIONS
    ])
    if not all_paths:
        raise FileNotFoundError(f"No images found in {data_dir}")

    random.seed(config.SEED)
    random.shuffle(all_paths)

    n      = len(all_paths)
    n_test = max(1, int(n * config.TEST_SPLIT))
    n_val  = max(1, int(n * config.VAL_SPLIT))

    train_paths = all_paths[:n - n_test - n_val]
    val_paths   = all_paths[n - n_test - n_val : n - n_test]
    test_paths  = all_paths[n - n_test:]

    print(f"Split — Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

    kw = dict(batch_size=batch_size,
              num_workers=config.NUM_WORKERS,
              pin_memory=config.PIN_MEMORY)
    return (
        DataLoader(CTDataset(train_paths, augment=True), shuffle=True,  **kw),
        DataLoader(CTDataset(val_paths),                 shuffle=False, **kw),
        DataLoader(CTDataset(test_paths),                shuffle=False, **kw),
    )
''')

# ── training/losses.py ────────────────────────────────────────────────────────
write('training/losses.py', '''
    import torch
    import torch.nn as nn
    import torchvision.models as tv_models

    class VGGFeatureExtractor(nn.Module):
        def __init__(self, feature_layer=18):
            super().__init__()
            vgg = tv_models.vgg19(weights=tv_models.VGG19_Weights.IMAGENET1K_V1)
            self.features = nn.Sequential(
                *list(vgg.features.children())[:feature_layer + 1]
            )
            for p in self.parameters():
                p.requires_grad = False
            self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
            self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

        def forward(self, x):
            if x.size(1) == 1:
                x = x.repeat(1, 3, 1, 1)
            x = (x - self.mean) / self.std
            return self.features(x)

    class PerceptualLoss(nn.Module):
        def __init__(self, feature_layer=18):
            super().__init__()
            self.vgg       = VGGFeatureExtractor(feature_layer)
            self.criterion = nn.MSELoss()
        def forward(self, pred, target):
            return self.criterion(self.vgg(pred), self.vgg(target))

    class GANLoss(nn.Module):
        def __init__(self, real_label=0.9, fake_label=0.0):
            super().__init__()
            self.real_val = real_label
            self.fake_val = fake_label
            self.loss     = nn.BCEWithLogitsLoss()
        def __call__(self, pred, is_real):
            val   = self.real_val if is_real else self.fake_val
            label = torch.full_like(pred, val)
            return self.loss(pred, label)

    class GeneratorLoss(nn.Module):
        def __init__(self, lambda_pixel=1.0, lambda_vgg=6e-3, lambda_adv=1e-3):
            super().__init__()
            self.lambda_pixel    = lambda_pixel
            self.lambda_vgg      = lambda_vgg
            self.lambda_adv      = lambda_adv
            self.pixel_loss      = nn.L1Loss()
            self.perceptual_loss = PerceptualLoss()
            self.gan_loss        = GANLoss()

        def forward(self, pred, target, disc_fake):
            l_pix  = self.pixel_loss(pred, target)
            l_perc = self.perceptual_loss(pred, target)
            l_adv  = self.gan_loss(disc_fake, True)
            total  = (self.lambda_pixel * l_pix
                    + self.lambda_vgg   * l_perc
                    + self.lambda_adv   * l_adv)
            return total, {"pixel": l_pix.item(),
                           "perceptual": l_perc.item(),
                           "adv": l_adv.item()}
''')

# ── training/train_cnn.py ─────────────────────────────────────────────────────
write('training/train_cnn.py', '''
    import sys
    sys.path.insert(0, '/kaggle/working/ct_denoising')

    import torch
    import torch.nn as nn
    from torch.optim.lr_scheduler import StepLR
    from pathlib import Path
    from tqdm import tqdm

    import cfg as config
    from models.cnn import DenoisingCNN
    from training.data_loader import get_dataloaders
    from utils.metrics import batch_psnr, batch_ssim

    def train_cnn(resume=False):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[CNN] device: {device}")

        train_loader, val_loader, _ = get_dataloaders()
        model     = DenoisingCNN(in_channels=config.NUM_CHANNELS).to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=config.CNN_LR)
        scheduler = StepLR(optimizer, step_size=config.CNN_LR_STEP, gamma=config.CNN_LR_GAMMA)

        start_epoch, best_psnr = 0, 0.0
        if resume and Path(config.CNN_CHECKPOINT).exists():
            ckpt = torch.load(config.CNN_CHECKPOINT, map_location=device)
            model.load_state_dict(ckpt["model"])
            optimizer.load_state_dict(ckpt["optimizer"])
            start_epoch = ckpt["epoch"] + 1
            best_psnr   = ckpt.get("psnr", 0.0)
            print(f"[CNN] Resumed from epoch {start_epoch}")

        for epoch in range(start_epoch, config.CNN_EPOCHS):
            model.train()
            train_loss = 0.0
            for noisy, clean in tqdm(train_loader,
                                     desc=f"CNN {epoch+1}/{config.CNN_EPOCHS}",
                                     leave=False):
                noisy, clean = noisy.to(device), clean.to(device)
                optimizer.zero_grad()
                loss = criterion(model(noisy), clean)
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            scheduler.step()

            model.eval()
            val_psnr = val_ssim = 0.0
            with torch.no_grad():
                for noisy, clean in val_loader:
                    noisy, clean = noisy.to(device), clean.to(device)
                    out = model(noisy)
                    val_psnr += batch_psnr(out, clean)
                    val_ssim += batch_ssim(out, clean)
            val_psnr /= len(val_loader)
            val_ssim /= len(val_loader)

            print(f"[CNN] epoch {epoch+1:3d}  "
                  f"loss {train_loss/len(train_loader):.4f}  "
                  f"PSNR {val_psnr:.2f}  SSIM {val_ssim:.4f}")

            is_best = val_psnr > best_psnr
            if is_best:
                best_psnr = val_psnr
            if (epoch + 1) % config.CNN_SAVE_EVERY == 0 or is_best:
                path = (config.CNN_CHECKPOINT if is_best
                        else f"{config.CHECKPOINT_DIR}/cnn_epoch{epoch+1}.pth")
                torch.save({"epoch": epoch, "model": model.state_dict(),
                            "optimizer": optimizer.state_dict(), "psnr": val_psnr}, path)

        print(f"[CNN] done. best PSNR: {best_psnr:.2f} dB")
        return model
''')

# ── training/train_gan.py ─────────────────────────────────────────────────────
write('training/train_gan.py', '''
    import sys
    sys.path.insert(0, '/kaggle/working/ct_denoising')

    import torch
    import torch.nn as nn
    from pathlib import Path
    from tqdm import tqdm

    import cfg as config
    from models.generator     import Generator
    from models.discriminator import Discriminator
    from training.losses      import GeneratorLoss, GANLoss, PerceptualLoss
    from training.data_loader import get_dataloaders
    from utils.metrics        import batch_psnr, batch_ssim

    WARMUP_EPOCHS = 10

    def train_gan(resume=False):
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[GAN] device: {device}")

        train_loader, val_loader, _ = get_dataloaders()

        G = Generator(in_channels=config.NUM_CHANNELS,
                      out_channels=config.NUM_CHANNELS).to(device)
        D = Discriminator(in_channels=config.NUM_CHANNELS).to(device)

        criterion_G = GeneratorLoss(config.LAMBDA_PIXEL,
                                    config.LAMBDA_VGG,
                                    config.LAMBDA_ADV).to(device)
        criterion_D = GANLoss()
        perc_loss   = PerceptualLoss().to(device)

        opt_G = torch.optim.Adam(G.parameters(),
                                 lr=config.GAN_LR_G,
                                 betas=(config.GAN_BETA1, config.GAN_BETA2))
        opt_D = torch.optim.Adam(D.parameters(),
                                 lr=config.GAN_LR_D,
                                 betas=(config.GAN_BETA1, config.GAN_BETA2))

        start_epoch, best_psnr = 0, 0.0

        if resume and Path(config.GEN_CHECKPOINT).exists():
            ckpt = torch.load(config.GEN_CHECKPOINT, map_location=device)
            G.load_state_dict(ckpt["model"])
            opt_G.load_state_dict(ckpt["optimizer"])
            start_epoch = ckpt.get("epoch", 0) + 1
            best_psnr   = ckpt.get("psnr", 0.0)
            print(f"[GAN] resumed generator from epoch {start_epoch}")

        if resume and Path(config.DIS_CHECKPOINT).exists():
            ckpt_d = torch.load(config.DIS_CHECKPOINT, map_location=device)
            D.load_state_dict(ckpt_d["model"])
            opt_D.load_state_dict(ckpt_d["optimizer"])
            print("[GAN] resumed discriminator")

        for epoch in range(start_epoch, config.GAN_EPOCHS):
            G.train(); D.train()
            warmup = epoch < WARMUP_EPOCHS
            loss_G_sum = loss_D_sum = 0.0

            desc = f"GAN {epoch+1}/{config.GAN_EPOCHS}" + (" [warmup]" if warmup else "")
            for noisy, clean in tqdm(train_loader, desc=desc, leave=False):
                noisy, clean = noisy.to(device), clean.to(device)

                opt_G.zero_grad()
                fake = G(noisy)
                if warmup:
                    loss_G = (config.LAMBDA_PIXEL * nn.L1Loss()(fake, clean)
                            + config.LAMBDA_VGG   * perc_loss(fake, clean))
                else:
                    loss_G, _ = criterion_G(fake, clean, D(fake))
                loss_G.backward()
                opt_G.step()
                loss_G_sum += loss_G.item()

                if not warmup:
                    opt_D.zero_grad()
                    loss_D = (criterion_D(D(clean),         True)
                            + criterion_D(D(fake.detach()), False)) * 0.5
                    loss_D.backward()
                    opt_D.step()
                    loss_D_sum += loss_D.item()

            G.eval()
            val_psnr = val_ssim = 0.0
            with torch.no_grad():
                for noisy, clean in val_loader:
                    noisy, clean = noisy.to(device), clean.to(device)
                    out = G(noisy)
                    val_psnr += batch_psnr(out, clean)
                    val_ssim += batch_ssim(out, clean)
            val_psnr /= len(val_loader)
            val_ssim /= len(val_loader)

            n_batch = len(train_loader)
            print(f"[GAN] epoch {epoch+1:3d}  "
                  f"G {loss_G_sum/n_batch:.4f}  "
                  f"D {loss_D_sum/n_batch:.4f}  "
                  f"PSNR {val_psnr:.2f}  SSIM {val_ssim:.4f}")

            is_best = val_psnr > best_psnr
            if is_best:
                best_psnr = val_psnr
            if (epoch + 1) % config.GAN_SAVE_EVERY == 0 or is_best:
                sfx = f"_epoch{epoch+1}"
                torch.save(
                    {"epoch": epoch, "model": G.state_dict(),
                     "optimizer": opt_G.state_dict(), "psnr": val_psnr},
                    config.GEN_CHECKPOINT if is_best
                    else f"{config.CHECKPOINT_DIR}/generator{sfx}.pth"
                )
                torch.save(
                    {"epoch": epoch, "model": D.state_dict(),
                     "optimizer": opt_D.state_dict()},
                    config.DIS_CHECKPOINT if is_best
                    else f"{config.CHECKPOINT_DIR}/discriminator{sfx}.pth"
                )

        print(f"[GAN] done. best PSNR: {best_psnr:.2f} dB")
        return G
''')

# ── utils/__init__.py ─────────────────────────────────────────────────────────
write('utils/__init__.py', '''
    from .metrics   import batch_psnr, batch_ssim, evaluate_model
    from .visualise import save_comparison_grid, save_metrics_bar_chart
''')

# ── utils/metrics.py ──────────────────────────────────────────────────────────
write('utils/metrics.py', '''
    import torch
    import numpy as np
    from skimage.metrics import structural_similarity as sk_ssim

    def batch_psnr(pred, target, max_val=1.0):
        mse = torch.clamp(torch.mean((pred - target) ** 2, dim=[1, 2, 3]), min=1e-10)
        return (10 * torch.log10(torch.tensor(max_val ** 2) / mse)).mean().item()

    def batch_ssim(pred, target):
        def to_np(t):
            return t.detach().cpu().permute(0, 2, 3, 1).numpy()
        scores = []
        for p, t in zip(to_np(pred), to_np(target)):
            p2 = p[..., 0] if p.shape[-1] == 1 else p
            t2 = t[..., 0] if t.shape[-1] == 1 else t
            scores.append(sk_ssim(p2, t2, data_range=1.0))
        return float(np.mean(scores))

    def evaluate_model(model, loader, device):
        model.eval()
        total_psnr = total_ssim = 0.0
        with torch.no_grad():
            for noisy, clean in loader:
                noisy, clean = noisy.to(device), clean.to(device)
                out = model(noisy)
                total_psnr += batch_psnr(out, clean)
                total_ssim += batch_ssim(out, clean)
        return {"psnr": total_psnr / len(loader),
                "ssim": total_ssim / len(loader)}
''')

# ── utils/visualise.py ────────────────────────────────────────────────────────
write('utils/visualise.py', '''
    import sys
    sys.path.insert(0, '/kaggle/working/ct_denoising')

    import numpy as np
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from pathlib import Path
    import cfg as config

    def _to_np(t):
        a = t.detach().cpu().numpy()
        return a[0] if a.shape[0] == 1 else np.transpose(a, (1, 2, 0))

    def save_comparison_grid(noisy, clean, cnn_out, gan_out,
                             save_path=None, n_samples=None):
        save_path = save_path or config.COMPARISON_PLOT
        n_samples = n_samples or config.NUM_VISUAL_SAMPLES
        n = min(n_samples, noisy.size(0))

        fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
        fig.suptitle("CT Denoising — Visual Comparison", fontsize=16, fontweight="bold")
        if n == 1:
            axes = axes[np.newaxis, :]

        for col, title in enumerate(["Clean (GT)", "Noisy Input", "CNN", "GAN"]):
            axes[0, col].set_title(title, fontsize=12, fontweight="bold")

        for row in range(n):
            for col, src in enumerate([clean, noisy, cnn_out, gan_out]):
                axes[row, col].imshow(_to_np(src[row]), cmap="gray", vmin=0, vmax=1)
                axes[row, col].axis("off")

        plt.tight_layout()
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Comparison saved \u2192 {save_path}")

    def save_metrics_bar_chart(cnn_m, gan_m, save_path=None):
        save_path = save_path or str(Path(config.RESULTS_DIR) / "metrics_bar.png")
        labels = ["PSNR (dB)", "SSIM"]
        x = np.arange(2)
        w = 0.35

        fig, ax = plt.subplots(figsize=(7, 5))
        b1 = ax.bar(x - w/2, [cnn_m["psnr"], cnn_m["ssim"]], w, label="CNN", color="#4c72b0")
        b2 = ax.bar(x + w/2, [gan_m["psnr"], gan_m["ssim"]], w, label="GAN", color="#dd8452")
        ax.set_title("CNN vs GAN Metrics", fontsize=14)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=12)
        ax.legend()
        ax.bar_label(b1, fmt="%.3f", padding=3)
        ax.bar_label(b2, fmt="%.3f", padding=3)

        plt.tight_layout()
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Bar chart saved \u2192 {save_path}")
''')

# ── testing/__init__.py ───────────────────────────────────────────────────────
write('testing/__init__.py', '')

# ── testing/evaluate.py ───────────────────────────────────────────────────────
write('testing/evaluate.py', '''
    import sys, csv, torch
    sys.path.insert(0, '/kaggle/working/ct_denoising')

    from pathlib import Path
    from tqdm import tqdm

    import cfg as config
    from models.cnn           import DenoisingCNN
    from models.generator     import Generator
    from training.data_loader import get_dataloaders
    from utils.metrics        import batch_psnr, batch_ssim
    from utils.visualise      import save_comparison_grid, save_metrics_bar_chart

    def load_model(model, path, device):
        if not Path(path).exists():
            raise FileNotFoundError(f"Checkpoint not found: {path}")
        ckpt = torch.load(path, map_location=device)
        model.load_state_dict(ckpt.get("model", ckpt))
        model.eval()
        print(f"  loaded: {path}")
        return model

    def evaluate():
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        _, _, test_loader = get_dataloaders()

        cnn = load_model(
            DenoisingCNN(in_channels=config.NUM_CHANNELS).to(device),
            config.CNN_CHECKPOINT, device
        )
        gan = load_model(
            Generator(in_channels=config.NUM_CHANNELS,
                      out_channels=config.NUM_CHANNELS).to(device),
            config.GEN_CHECKPOINT, device
        )

        cp = cs = gp = gs = np_ = ns_ = 0.0
        vis = [None] * 4

        with torch.no_grad():
            for i, (noisy, clean) in enumerate(tqdm(test_loader, desc="Evaluating")):
                noisy, clean = noisy.to(device), clean.to(device)
                oc = cnn(noisy)
                og = gan(noisy)
                cp  += batch_psnr(oc,    clean)
                cs  += batch_ssim(oc,    clean)
                gp  += batch_psnr(og,    clean)
                gs  += batch_ssim(og,    clean)
                np_ += batch_psnr(noisy, clean)
                ns_ += batch_ssim(noisy, clean)
                if i == 0:
                    vis = [noisy.cpu(), clean.cpu(), oc.cpu(), og.cpu()]

        n = len(test_loader)
        noisy_m = {"psnr": np_ / n, "ssim": ns_ / n}
        cnn_m   = {"psnr": cp  / n, "ssim": cs  / n}
        gan_m   = {"psnr": gp  / n, "ssim": gs  / n}

        print(f"\n{chr(61)*52}")
        print(f"{'Model':<18} {'PSNR':>10} {'SSIM':>10}")
        print(f"{chr(45)*52}")
        for name, m in [("Noisy", noisy_m), ("CNN", cnn_m), ("GAN", gan_m)]:
            print(f"{name:<18} {m['psnr']:>10.2f} {m['ssim']:>10.4f}")
        print(f"{chr(61)*52}\n")

        Path(config.RESULTS_DIR).mkdir(parents=True, exist_ok=True)
        with open(config.RESULTS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["model", "psnr", "ssim"])
            w.writeheader()
            for name, m in [("Noisy", noisy_m), ("CNN", cnn_m), ("GAN", gan_m)]:
                w.writerow({"model": name, **m})

        save_comparison_grid(*vis)
        save_metrics_bar_chart(cnn_m, gan_m)
        return cnn_m, gan_m
''')

print('All project files written.')
print()
for root, dirs, files in os.walk(PROJECT):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    level   = root.replace(PROJECT, '').count(os.sep)
    indent  = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

## Cell 4 — Sanity Check

In [ ]:
import os, sys, glob
sys.path.insert(0, '/kaggle/working/ct_denoising')

for f in ['cfg.py', 'models/generator.py', 'training/train_cnn.py']:
    path = f'/kaggle/working/ct_denoising/{f}'
    print('OK' if os.path.exists(path) else 'MISSING', path)

import cfg as config
print('\nDATA_DIR:', config.DATA_DIR)
print('exists:', os.path.exists(config.DATA_DIR))
if os.path.exists(config.DATA_DIR):
    imgs = glob.glob(os.path.join(config.DATA_DIR, '**', '*.png'), recursive=True)
    print('Total PNG count:', len(imgs))
    if imgs:
        print('Sample image:', imgs[0])

## Cell 5 — Train CNN Baseline

In [ ]:
import os, gc, sys

os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_DISABLE_DYNAMO"] = "1"  
os.environ["TORCH_COMPILE_DISABLE"] = "1"

# ── CRITICAL: import torch FULLY before touching sys.path ─────────────────────
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
# Force all torch internals to load NOW, before sys.path is modified
_ = torch.optim.Adam([torch.zeros(1, requires_grad=True)], lr=1e-3)
print("Torch internals loaded OK")

# ── NOW it is safe to add the project to sys.path ─────────────────────────────
PROJECT = '/kaggle/working/ct_denoising'

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['cfg', 'models', 'training', 'utils', 'testing']):
        del sys.modules[mod]
gc.collect()

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

# Delete old config.py if it somehow still exists
old = os.path.join(PROJECT, 'config.py')
if os.path.exists(old):
    os.remove(old)
    print("Deleted stale config.py")

import cfg as config
config.CNN_EPOCHS = 60
config.BATCH_SIZE = 4

from training.train_cnn import train_cnn
resume = os.path.exists(config.CNN_CHECKPOINT)
print('Resuming from checkpoint.' if resume else 'Starting fresh.')
cnn_model = train_cnn(resume=resume)

## Cell 6 — Train GAN

First 10 epochs are pixel+perceptual warmup (no discriminator). Adversarial loss kicks in from epoch 11.

In [ ]:
import gc, sys, os, torch

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['cfg', 'models', 'training', 'utils', 'testing']):
        del sys.modules[mod]
gc.collect()
torch.cuda.empty_cache()

sys.path.insert(0, '/kaggle/working/ct_denoising')
os.chdir('/kaggle/working/ct_denoising')

import cfg as config
config.GAN_EPOCHS = 55
config.BATCH_SIZE = 4

from training.train_gan import train_gan
resume = os.path.exists(config.GEN_CHECKPOINT)
print('Resuming from checkpoint.' if resume else 'Starting fresh.')
gan_model = train_gan(resume=resume)

## Cell 7 — Evaluate and Plot Results

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/ct_denoising')
os.chdir('/kaggle/working/ct_denoising')

from testing.evaluate import evaluate
cnn_metrics, gan_metrics = evaluate()